# 🧠 Entraînement IA — Ultimate TTT

Ce notebook génère les données d'entraînement, entraîne le modèle,
et le télécharge prêt à être mis dans ton repo GitHub.

**Durée totale : ~20 minutes avec GPU**

### Avant de lancer :
1. **Activer le GPU** : Menu `Exécution` → `Modifier le type d'exécution` → GPU T4
2. Exécuter les cellules dans l'ordre (**Shift+Entrée**)

## 📦 Étape 1 — Installer tensorflowjs

In [ ]:
!pip install tensorflowjs -q
print('✅ tensorflowjs installé')

## 🎮 Étape 2 — Logique du jeu + encodage

In [ ]:
import numpy as np
import random
from copy import deepcopy

LINES = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]

def new_state():
    return {'boards':[[None]*9 for _ in range(9)],'bw':[None]*9,'active':None,'player':'X','winner':None}

def check_result(cells):
    for a,b,c in LINES:
        if cells[a] and cells[a]==cells[b]==cells[c]: return cells[a]
    return 'draw' if all(cells) else None

def apply_move(state, b, c):
    ns = {'boards':[r[:] for r in state['boards']],'bw':state['bw'][:],'active':state['active'],'player':state['player'],'winner':state['winner']}
    ns['boards'][b][c] = ns['player']
    br = check_result(ns['boards'][b])
    if br: ns['bw'][b] = br
    gr = check_result(ns['bw'])
    if gr: ns['winner'] = gr
    ns['active'] = c if not ns['bw'][c] else None
    ns['player'] = 'O' if ns['player']=='X' else 'X'
    return ns

def get_moves(state):
    moves = []
    for b in range(9):
        if state['bw'][b]: continue
        if state['active'] is not None and state['active']!=b: continue
        for c in range(9):
            if not state['boards'][b][c]: moves.append((b,c))
    return moves

def encode_state(state):
    f = []
    for b in range(9):
        for c in range(9):
            v = state['boards'][b][c]
            f += [1,0,0] if v=='X' else [0,1,0] if v=='O' else [0,0,1]
    for b in range(9):
        v = state['bw'][b]
        f += [1,0,0] if v=='X' else [0,1,0] if v=='O' else [0,0,1]
    act = state['active']
    for i in range(9): f.append(1 if act==i else 0)
    f.append(1 if act is None else 0)
    f.append(1 if state['player']=='O' else 0)
    return np.array(f, dtype=np.float32)

print(f'✅ Encodage : {len(encode_state(new_state()))} features')

## 📊 Étape 3 — Fonction d'évaluation + minimax

In [ ]:
def eval_state(state):
    score = 0
    bw, boards = state['bw'], state['boards']
    for b in range(9):
        if bw[b]=='O': score+=100
        elif bw[b]=='X': score-=100
    for a,b,c in LINES:
        t=[bw[a],bw[b],bw[c]]
        p=t.count('O');x=t.count('X');n=t.count(None)
        if p==2 and n==1: score+=50
        if x==2 and n==1: score-=50
        if p==1 and n==2: score+=10
        if x==1 and n==2: score-=10
    if bw[4]=='O': score+=30
    elif bw[4]=='X': score-=30
    for co in [0,2,6,8]:
        if bw[co]=='O': score+=15
        elif bw[co]=='X': score-=15
    for b in range(9):
        if bw[b]: continue
        for a,c,d in LINES:
            cells=[boards[b][a],boards[b][c],boards[b][d]]
            p=cells.count('O');x=cells.count('X');n=cells.count(None)
            if p==2 and n==1: score+=8
            if x==2 and n==1: score-=8
            if p==1 and n==2: score+=2
            if x==1 and n==2: score-=2
        if boards[b][4]=='O': score+=4
        elif boards[b][4]=='X': score-=4
    return score

def minimax(state, depth, is_max, alpha, beta):
    if state['winner']=='O': return 1000
    if state['winner']=='X': return -1000
    if state['winner']=='draw': return 0
    if depth==0: return eval_state(state)
    moves = get_moves(state)
    if not moves: return 0
    if is_max:
        v=-float('inf')
        for b,c in moves:
            v=max(v,minimax(apply_move(state,b,c),depth-1,False,alpha,beta))
            alpha=max(alpha,v)
            if beta<=alpha: break
        return v
    else:
        v=float('inf')
        for b,c in moves:
            v=min(v,minimax(apply_move(state,b,c),depth-1,True,alpha,beta))
            beta=min(beta,v)
            if beta<=alpha: break
        return v

print('✅ Minimax prêt')

## 🎲 Étape 4 — Génération des données (~15 min)

In [ ]:
N_GAMES = 10000
MAX_DEPTH = 4

def play_random_game():
    state = new_state()
    states = []
    while not state['winner']:
        moves = get_moves(state)
        if not moves: break
        states.append(deepcopy(state))
        b,c = random.choice(moves)
        state = apply_move(state,b,c)
    return states

def label_state(state):
    moves = get_moves(state)
    if not moves: return 0.0
    best = -float('inf') if state['player']=='O' else float('inf')
    for b,c in moves:
        sc = minimax(apply_move(state,b,c),MAX_DEPTH-1,state['player']=='X',-float('inf'),float('inf'))
        best = max(best,sc) if state['player']=='O' else min(best,sc)
    return float(best)

print(f'Génération de {N_GAMES} parties...')
X_list, y_list = [], []
for i in range(N_GAMES):
    if i%1000==0: print(f'  {i}/{N_GAMES}...')
    states = play_random_game()
    for s in random.sample(states, min(5,len(states))):
        X_list.append(encode_state(s))
        y_list.append(np.clip(label_state(s)/1000.0,-1.0,1.0))

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.float32)
np.savez('training_data.npz', X=X, y=y)
print(f'\n✅ {len(X)} exemples générés et sauvegardés')

## 🚀 Étape 5 — Entraînement (~5 min avec GPU)

In [ ]:
import tensorflow as tf
from tensorflow import keras
print(f'TensorFlow {tf.__version__} — GPU: {tf.config.list_physical_devices("GPU")}')

model = keras.Sequential([
    keras.layers.Input(shape=(X.shape[1],)),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(1, activation='tanh')
], name='ultimate_ttt')

model.compile(optimizer=keras.optimizers.Adam(0.001), loss='mse', metrics=['mae'])
model.summary()

split = int(len(X)*0.9)
history = model.fit(
    X[:split], y[:split],
    validation_data=(X[split:], y[split:]),
    epochs=50, batch_size=256,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, verbose=1)
    ],
    verbose=1
)
loss, mae = model.evaluate(X[split:], y[split:], verbose=0)
print(f'\n✅ Val loss={loss:.4f} | MAE={mae:.4f}')

## 📦 Étape 6 — Export TensorFlow.js + téléchargement

In [ ]:
import tensorflowjs as tfjs
import os, shutil
from google.colab import files

os.makedirs('model', exist_ok=True)
tfjs.converters.save_keras_model(model, 'model')
print('✅ Modèle exporté dans model/')
print(os.listdir('model'))

# Zip le dossier model/ pour téléchargement
shutil.make_archive('model', 'zip', '.', 'model')
files.download('model.zip')
print('\n✅ Téléchargement de model.zip lancé !')
print('\nProchaines étapes :')
print('1. Dézippe model.zip')
print('2. Copie le dossier model/ dans ~/Ultimate_TTT_fix/')
print('3. git add model/ && git commit -m "feat: trained AI model" && git push')